In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer - Una función de Qiskit por Kipu Quantum

> **Note:** * Las funciones de Qiskit son una característica experimental disponible únicamente para los usuarios de los planes IBM Quantum&reg; Premium, Flex y On-Prem (a través de la API de IBM Quantum Platform). Se encuentran en estado de versión preliminar y están sujetas a cambios.

## Descripción general
Con el Iskay Quantum Optimizer de Kipu Quantum, puedes abordar problemas de optimización complejos utilizando los computadores cuánticos de IBM&reg;. Este solver aprovecha el algoritmo [bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) de vanguardia de Kipu, que requiere únicamente la función objetivo como entrada para entregar automáticamente las soluciones a los problemas. Puede gestionar problemas de optimización que involucran hasta 156 qubits, lo que permite utilizar todos los qubits de los dispositivos cuánticos de IBM. El Optimizer emplea un mapeo 1 a 1 entre variables clásicas y qubits, lo que te permite abordar problemas de optimización con hasta 156 variables binarias.

El Optimizer permite resolver problemas de optimización binaria sin restricciones. Además de la formulación QUBO (Quadratic Unconstrained Binary Optimization) de uso común, también admite problemas de optimización de orden superior (HUBO). El solver utiliza un algoritmo cuántico no variacional, realizando la mayor parte del cómputo en dispositivos cuánticos.

A continuación se presentan más detalles sobre el algoritmo utilizado y una guía breve sobre cómo usar la función, así como resultados de benchmarking en diversas instancias de problemas de diferentes tamaños y complejidades.

## Descripción
El Optimizer es una implementación lista para usar de algoritmos de optimización cuántica de vanguardia. Resuelve problemas de optimización ejecutando circuitos cuánticos altamente comprimidos en hardware cuántico. Esta compresión se logra introduciendo términos contradiabáticos en la evolución temporal subyacente del sistema cuántico. El algoritmo ejecuta varias iteraciones de ejecuciones en hardware para obtener las soluciones finales y las combina con posprocesamiento. Estos pasos están integrados de forma transparente en el flujo de trabajo del Optimizer y se ejecutan automáticamente.

### ¿Cómo funciona el Quantum Optimizer?
Esta sección describe los fundamentos del algoritmo bf-DCQO implementado. También se puede encontrar una introducción al algoritmo en el [canal de YouTube de Qiskit.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

El algoritmo se basa en la evolución temporal de un sistema cuántico que se transforma con el tiempo, donde la solución al problema está codificada en el estado fundamental del sistema cuántico al final de la evolución. Según el [teorema adiabático](https://en.wikipedia.org/wiki/Adiabatic_theorem), esta evolución debe ser lenta para garantizar que el sistema permanezca en su estado fundamental. Digitalizar esta evolución es la base del cómputo adiabático cuántico digitalizado (DQA) y del conocido algoritmo QAOA. Sin embargo, la evolución lenta requerida no es factible a medida que aumenta el tamaño del problema, ya que resulta en una profundidad de circuito creciente. Al usar protocolos contradiabáticos, es posible suprimir las excitaciones no deseadas que ocurren durante tiempos de evolución cortos, permaneciendo en el estado fundamental. En este caso, digitalizar este tiempo de evolución más corto produce circuitos cuánticos de menor profundidad y con menos puertas de entrelazamiento.

Los circuitos de los algoritmos bf-DCQO típicamente usan hasta diez veces menos puertas de entrelazamiento que DQA, y de tres a cuatro veces menos puertas de entrelazamiento que las implementaciones estándar de QAOA. Gracias al menor número de puertas, ocurren menos errores durante la ejecución del circuito en hardware. Por tanto, el optimizer no requiere el uso de técnicas como supresión de errores o mitigación de errores. Implementarlas en versiones futuras puede mejorar aún más la calidad de las soluciones.

Aunque el algoritmo bf-DCQO usa iteraciones, es no variacional. Después de cada iteración del algoritmo, se mide la distribución de estados. La distribución obtenida se usa para calcular un llamado campo de sesgo (bias-field). El campo de sesgo permite comenzar la siguiente iteración desde un estado de energía cercano a la solución encontrada anteriormente. De esta manera, el algoritmo avanza con cada iteración hacia soluciones de menor energía. Típicamente, aproximadamente diez iteraciones son suficientes para converger a una solución, requiriendo en total un número de iteraciones mucho menor que los algoritmos variacionales, que es del orden de aproximadamente 100 iteraciones.

El optimizer combina el algoritmo bf-DCQO con posprocesamiento clásico. Después de medir la distribución de estados, se realiza una búsqueda local. Durante la búsqueda local, los bits de la solución medida se invierten aleatoriamente. Después de la inversión, se evalúa la energía del nuevo bitstring. Si la energía es menor, el bitstring se conserva como la nueva solución. La búsqueda local solo escala linealmente con el número de qubits; por lo tanto, es computacionalmente económica. Dado que el posprocesamiento corrige inversiones de bits locales, compensa los errores de inversión de bits que a menudo son el resultado de imperfecciones del hardware y errores de lectura.

### Flujo de trabajo
A continuación se muestra un esquema del flujo de trabajo del Quantum Optimizer.

![Flujo de trabajo](../docs/images/guides/kipu-optimization/workflow.svg "Flujo de trabajo del Quantum Optimizer")

Al utilizar el Quantum Optimizer, resolver un problema de optimización en hardware cuántico se puede reducir a:
* Formular la función objetivo del problema
* Acceder al Optimizer a través de las funciones de Qiskit
* Ejecutar el Optimizer y recopilar el resultado

## Benchmarks
Las métricas de benchmark a continuación muestran que el Optimizer aborda eficazmente problemas de hasta 156 qubits y ofrecen una visión general de la precisión y escalabilidad del optimizer en diferentes tipos de problemas. Ten en cuenta que las métricas de rendimiento reales pueden variar según las características específicas del problema, como el número de variables, la densidad y localidad de los términos en la función objetivo, y el orden polinomial.

La siguiente tabla incluye la razón de aproximación (AR), una métrica definida de la siguiente manera:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
donde $C$ es la función objetivo, $C_{\textrm{min}}$, $C_{\textrm{max}}$ son sus valores mínimo y máximo, y $C^{*}$ es el costo de la mejor solución encontrada, respectivamente. Por lo tanto, AR=100\% significa que se ha obtenido el estado fundamental del problema.

| Ejemplo           | Número de qubits | Razón de aproximación | Tiempo total (s) | Uso de runtime (s) | Número total de shots | Número de iteraciones |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| MaxCut no ponderado |        28        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut no ponderado |        30        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut no ponderado |        32        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut no ponderado |        80        |   100%   |      480       |        60         |          90k          |          9           |
| MaxCut no ponderado |       100        |   100%   |      330       |        60         |          60k          |          6           |
| MaxCut no ponderado |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- Las instancias de MaxCut con 28, 30 y 32 qubits se ejecutaron en ibm_sherbrooke. Las instancias con 80, 100 y 120 se ejecutaron en un procesador Heron r2.
- Las instancias HUBO también se ejecutaron en un procesador Heron r2.

Todas las instancias de benchmark están disponibles en GitHub (ver [instancias de benchmark de Kipu](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Un ejemplo para ejecutar estas instancias se puede encontrar en [Ejemplo 3: Instancias de benchmark](#example-3-benchmark-instances).

## Entradas y salidas
### Entrada
Consulta la siguiente tabla para conocer todos los parámetros de entrada que acepta el Quantum Optimizer. La sección **Opciones** a continuación proporciona más detalles sobre las `options` disponibles.

| Nombre         | Tipo               | Descripción                                                                                                                                                                            | Requerido | Predeterminado                                                                                       | Ejemplo                                                                                                              |
| ------------ | ------------------ | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------- | --------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | Los coeficientes del problema de optimización formulado en formato QUBO/HUBO o spin. Para más información sobre la especificación del problema, consulta [Formatos de problema aceptados](#accepted-problem-formats)                                       | Sí      | N/A                                                                                           | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                             |
| problem_type | `str`              | Especifica si los coeficientes del problema están en formato binario (QUBO/HUBO) o spin. Las dos posibilidades son `"spin"` o `"binary"`                                                   | Sí      | N/A                                                                                           | `"spin"`                                                                                                             |
| backend_name | `str`              | Nombre del backend para realizar la consulta                                                                                                                                                  | Sí       | N/A                                                      | `"ibm_fez"`                                                                                                          |                                                                                         |
| options      | `Dict[str, Any]`   | Opciones para gestionar el envío al hardware, como el número de shots. Para más detalles sobre la configuración de opciones, consulta la sección [Opciones](#options) | No       | Para ver los valores predeterminados de la configuración de opciones, consulta la sección [Opciones](#options)                                                         | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}` |

#### Formatos de problema aceptados
Los argumentos `problem` y `problem_type` codifican un problema de optimización de la forma

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
donde

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- Al elegir `problem_type = "binary"`, especificas que la función de costo está en formato `binary`, lo que significa que $D = {0,  1}^{n}$, es decir, la función de costo está escrita en formulación QUBO/HUBO.
- Por otro lado, al elegir `problem_type = "spin"`, la función de costo está escrita en formulación de Ising, donde $D = {-1, 1}^{n}$.

Los coeficientes del problema deben codificarse en un diccionario de la siguiente manera:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- Ten en cuenta que las claves del diccionario deben ser cadenas que contengan una tupla válida de enteros no repetidos.

#### Opciones
Iskay ofrece capacidades de ajuste fino a través de parámetros opcionales. Aunque los valores predeterminados funcionan bien para la mayoría de los problemas, puedes personalizar el comportamiento para requisitos específicos:

| Parámetro | Tipo | Predeterminado | Descripción |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | Mediciones cuánticas por iteración (mayor = más preciso) |
| **num_iterations** | `int` | 10 | Iteraciones del algoritmo (más iteraciones pueden mejorar la calidad de la solución) |
| **use_session** | `bool` | True | Usar sesiones de IBM para reducir los tiempos de espera en cola |
| **seed_transpiler** | `int` | None | Establecer para una compilación reproducible del circuito cuántico |
| **direct_qubit_mapping** | `bool` | False | Mapear qubits virtuales directamente a qubits físicos |
| **job_tags** | `List[str]` | None | Etiquetas personalizadas para el seguimiento de trabajos |
| **preprocessing_level** | `int` | 0 | Intensidad del preprocesamiento del problema (0-3) — ver detalles a continuación |
| **postprocessing_level** | `int` | 2 | Nivel de refinamiento de la solución (0-2) — ver detalles a continuación |
| **transpilation_level** | `int` | 0 | Intentos de optimización del transpilador (0-5) — ver detalles a continuación |
| **transpile_only** | `bool` | False | Analizar la optimización del circuito sin ejecutar la ejecución completa |

**Niveles de preprocesamiento (0-3)**: Especialmente importantes para problemas más grandes que actualmente no caben en los tiempos de coherencia del hardware. Los niveles de preprocesamiento más altos logran profundidades de circuito más superficiales mediante aproximaciones en la transpilación del problema:
- **Nivel 0**: Exacto, circuitos más largos
- **Nivel 1**: Buen equilibrio entre precisión y aproximación, eliminando solo las puertas con ángulos en el percentil 10 más bajo
- **Nivel 2**: Aproximación ligeramente mayor, eliminando las puertas con ángulos en el percentil 20 más bajo y usando `approximation_degree=0.95` en la transpilación
- **Nivel 3**: Nivel máximo de aproximación, eliminando las puertas en el percentil 30 más bajo y usando `approximation_degree=0.90` en la transpilación

**Niveles de transpilación (0-5)**: Controlan los intentos avanzados de optimización del transpilador para la compilación de circuitos cuánticos. Esto puede provocar un aumento en la sobrecarga clásica y, en algunos casos, puede no cambiar la profundidad del circuito. El valor predeterminado `2` generalmente produce el circuito más pequeño y es relativamente rápido.
- **Nivel 0**: Optimización del circuito DCQO descompuesto (layout, enrutamiento, programación)
- **Nivel 1**: Optimización de `PauliEvolutionGate` y luego del circuito DCQO descompuesto (max_trials=10)
- **Nivel 2**: Optimización de `PauliEvolutionGate` y luego del circuito DCQO descompuesto (max_trials=15)
- **Nivel 3**: Optimización de `PauliEvolutionGate` y luego del circuito DCQO descompuesto (max_trials=20)
- **Nivel 4**: Optimización de `PauliEvolutionGate` y luego del circuito DCQO descompuesto (max_trials=25)
- **Nivel 5**: Optimización de `PauliEvolutionGate` y luego del circuito DCQO descompuesto (max_trials=50)

**Niveles de posprocesamiento (0-2)**: Controlan cuánta optimización clásica se aplica, compensando los errores de inversión de bits con diferentes números de pasadas greedy de una búsqueda local:
- **Nivel 0**: 1 pasada
- **Nivel 1**: 2 pasadas
- **Nivel 2**: 3 pasadas

**Modo solo transpilación**: Ahora disponible para usuarios que deseen analizar la optimización del circuito sin ejecutar la ejecución completa del algoritmo cuántico.

**Ejemplo de configuración personalizada**: Así es como podrías configurar Iskay con diferentes ajustes:

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Optimización de la semilla**: Ten en cuenta que `seed_transpiler` se establece en `None` de forma predeterminada. Esto activa el proceso de optimización automática del transpilador. Cuando es `None`, el sistema iniciará una prueba con múltiples semillas y seleccionará la que produzca la menor profundidad de circuito, aprovechando al máximo el parámetro `max_trials` para cada nivel de transpilación.

**Rendimiento del nivel de transpilación**: Aumentar el número de `max_trials` con valores más altos para `transpilation_level` inevitablemente incrementará el tiempo de transpilación, pero puede que no siempre cambie el circuito final; esto depende en gran medida de la estructura y complejidad específicas del circuito. Sin embargo, para algunos circuitos/problemas, la diferencia entre 10 intentos (nivel 1) y 50 intentos (nivel 5) puede ser considerable, por lo que explorar estos parámetros puede ser clave para encontrar una solución con éxito.

### Salida
| Nombre   | Tipo               | Descripción                                                           | Ejemplo                                                                |
| ------ | ------------------ | ----------------------------------------------------------------------| ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | Solución y metadatos. La estructura varía según la opción `transpile_only`. |   Ver "Contenido del diccionario de resultados" a continuación      |

### Contenido del diccionario de resultados
La estructura del diccionario de resultados depende del modo de ejecución:

| Campo         | Tipo               | Modo | Descripción                                                                                                             | Ejemplo |
| ------------ | :------------------: | :--: | ----------------------------------------------------------------------------------------------------------------------- | ------- |
| **solution**     | `Dict[str, int]` | **Estándar** | La solución mapeada ordenada, donde las claves son los índices de variables (como cadenas) ordenados numéricamente y los valores son los correspondientes valores de variables (1/-1 para problemas spin, 1/0 para problemas binarios).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **Estándar** | Información detallada sobre la solución (ver detalles a continuación)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **Estándar** | El tipo de problema de optimización ('spin' o 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Solo transpilación** | Análisis del circuito y detalles de transpilación (ver detalles a continuación) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### Ejecución estándar
Cuando el parámetro opcional `transpile_only=False`:

**Diccionario solution_info:**
- **"bitstring"** (`str`): La representación en bitstring sin procesar de la solución.
- **"cost"** (`float`): El valor de costo/energía asociado con la solución.
- **"seed_transpiler"** (`int`): La semilla aleatoria utilizada para el transpilador que produjo este resultado.
- **"mapping"** (`Dict[int, int]`): El mapeo original qubit-a-variable usado en el cómputo.
- **"qpu_time"** (`float`, opcional): El tiempo de ejecución en la QPU en segundos.

**Notas sobre el mapeo de variables:**
- El diccionario `solution` se obtiene del bitstring de solución, usando el objeto `mapping` para indexar las variables.
- Cuando `problem_type=spin`, se usa la asignación $1 \rightarrow -1, \quad 0 \rightarrow 1$.
- Las claves del diccionario de solución son índices de variables ordenados numéricamente como cadenas.

#### Análisis de transpilación
Cuando el parámetro opcional `transpile_only=True`:

**Diccionario transpilation_info:**
- **"best_seed"** (`int`): La semilla óptima encontrada para la transpilación
- **"transpilation_time_seconds"** (`float`): Tiempo empleado en el proceso de transpilación
- **"transpiled_circuit"** (`Dict`): Análisis del circuito que contiene:
  - **"depth"** (`int`): Profundidad del circuito (número de capas)
  - **"gate_count"** (`int`): Número total de puertas en el circuito
  - **"num_qubits"** (`int`): Número de qubits utilizados
  - **"width"** (`int`): Ancho del circuito
  - **"operations"** (`Dict[str, int]`): Recuento de cada tipo de puerta utilizada

**Uso del modo solo transpilación:**
- Disponible para usuarios que deseen analizar la optimización del circuito sin ejecutar la ejecución completa del algoritmo cuántico.
- Útil para el análisis de circuitos, estudios de optimización de profundidad y comprensión de los efectos de la transpilación antes de comprometerse con la ejecución completa.

## Primeros pasos
En esta documentación, repasaremos los pasos para usar el Iskay Quantum Optimizer. En el proceso, mostraremos rápidamente cómo cargar la función desde el catálogo y cómo convertir tu problema a una entrada válida, mostrando al mismo tiempo cómo puedes experimentar con diferentes parámetros opcionales.

Para un ejemplo más detallado, consulta el tutorial [Resuelve el problema Market Split con el Iskay Quantum Optimizer de Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), donde trabajamos todo el proceso de usar Iskay Solver para abordar el problema Market Split, que representa un desafío de asignación de recursos del mundo real donde los mercados deben particionarse en regiones de ventas equilibradas para cumplir con objetivos de demanda exactos.

Autentícate con tu clave de API, que puedes encontrar en el [panel de control de IBM Quantum Platform](http://quantum.cloud.ibm.com/), y selecciona la función de Qiskit de la siguiente manera:

In [ ]:
# ruff: noqa: F821

> **Note:** El siguiente código asume que has guardado tus credenciales. Si no lo has hecho, sigue las instrucciones en [guardar tu cuenta de IBM Cloud](/guides/functions#install-qiskit-functions-catalog-client) para autenticarte con tu clave de API.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Ejemplo 1: Función de costo simple

Considera la función de costo en formulación spin:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

donde $(x_0, ..., x_4) \in {-1, 1}^5$.

La solución a esta función de costo simple es
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
con valor mínimo $C^{*} = -6$

### 1. Crear la función objetivo

Comenzamos creando un diccionario con los coeficientes de la función objetivo de la siguiente manera:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Ejecutar el Optimizer
Resolvemos el problema ejecutando el optimizer. Dado que $(x_0, ..., x_4) \in {-1, 1}^5$, debemos establecer `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Obtener el resultado
La solución del problema de optimización se obtiene directamente desde el optimizer.

In [ ]:
print(job.result())

Esto mostrará un diccionario de la forma:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Observa que el diccionario `solution` muestra el vector resultado $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Ejemplo 2: MaxCut

Muchos problemas de grafos como MaxCut o el conjunto independiente máximo son problemas NP-difíciles e ideales para probar algoritmos cuánticos y hardware. Este ejemplo demuestra cómo resolver el problema MaxCut de un grafo 3-regular con el Quantum Optimizer.

Para ejecutar este ejemplo debes instalar el paquete `networkx` además de `qiskit-ibm-catalog`. Para instalarlo, ejecuta el siguiente comando:

In [ ]:
# %pip install networkx numpy

### 1. Crear la función objetivo
Comienza generando un grafo 3-regular aleatorio. Para este grafo, definimos la función objetivo del problema MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Ejecutar el Optimizer
Resuelve el problema ejecutando el optimizer.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Obtener el resultado
Recupera el resultado y mapea el bitstring de solución de vuelta a los nodos originales del grafo.

In [ ]:
print(job.result())

La solución al problema Maxcut está contenida directamente en el sub-diccionario `solution` del objeto resultado

In [ ]:
maxcut_solution = job.result()["solution"]

## Ejemplo 3: Instancias de benchmark
Las instancias de benchmark están disponibles en GitHub: [Instancias de benchmark de Kipu.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

Las instancias se pueden cargar usando la biblioteca `pygithub`. Para instalarla, ejecuta el siguiente comando:

In [ ]:
# %pip install pygithub

Las rutas para las instancias de benchmark son:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Para reproducir el rendimiento del benchmark para las instancias HUBO, selecciona el backend `ibm_marrakesh` y establece `direct_qubit_mapping` en `True` en el sub-diccionario `options`.
El siguiente ejemplo ejecuta la instancia Maxcut con 32 nodos.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Casos de uso
Los casos de uso típicos para el solver de optimización son problemas de optimización combinatoria. Puedes resolver problemas de muchas industrias como finanzas, farmacéutica o logística. A continuación se presentan algunos ejemplos.
* Optimización de cartera (QUBO): [publicación científica](https://doi.org/10.1103/PhysRevApplied.22.054037) y [libro blanco](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Plegamiento de proteínas (HUBO): [publicación científica](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Programación logística (QUBO): [publicación científica](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Optimización de redes: [webinar](https://www.youtube.com/watch?v=w5SrCIK88No)
* Market split (QUBO): [tutorial](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Si estás interesado en abordar un caso de uso específico y desarrollar un mapeo dedicado, podemos ayudarte. [Contáctanos.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)

## Obtener soporte
Para soporte, contacta a support@kipu-quantum.com.

## Próximos pasos
- [Solicitar acceso al Quantum Optimizer de Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- Prueba el tutorial [Resuelve el problema Market Split con el Iskay Quantum Optimizer de Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Consulta [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- Consulta [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- Consulta [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)

## Información adicional
Iskay, al igual que el nombre de nuestra empresa Kipu Quantum, es una palabra peruana. Aunque somos una startup de Alemania, estas palabras provienen del país natal de uno de nuestros cofundadores, donde el Quipu fue una de las primeras máquinas de cálculo desarrolladas por la humanidad hace 2000 años a.C.